In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

## Dataset 1: Instacart Analysis

In [2]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [9]:
X = data1_transformed.drop(columns = ['ordered'])
y = data1_transformed['ordered']

model = LinearRegression()

model.fit(X, y)

model.coef_

array([-1.85844815e-08,  3.30587125e-08,  8.43130152e-02,  1.22894456e-02,
        1.30557690e-02,  5.66384002e-04, -2.88704293e-02,  1.22894456e-02,
       -4.72387383e-02])

In [11]:
X.columns

Index(['user_id', 'product_id', 'log_times_purchased', 'log_num_orders',
       'log_frequency', 'log_Product Popularity', 'log_Last Product Order',
       'log_Last Order', 'log_Orders Since Last Purchase'],
      dtype='str')

In [14]:
cols = X.columns

X_squared = pd.concat((X, (X**2).rename(columns = {f'{var}' : f'{var}2' for var in cols })), axis = 1)
X_squared = sm.add_constant(X_squared)

X_squared.head()

,const,user_id,product_id,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase,user_id2,product_id2,log_times_purchased2,log_num_orders2,log_frequency2,log_Product Popularity2,log_Last Product Order2,log_Last Order2,log_Orders Since Last Purchase2
0,1.0,71,45,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050,5041,2025,3.210402,10.100026,0.038695,0.000035,5.301898,10.100026,7.333536
1,1.0,71,117,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000,5041,13689,8.974412,10.100026,0.362615,0.000001,10.100026,10.100026,0.000000
2,1.0,71,2078,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522,5041,4318084,0.480453,10.100026,0.001811,0.000042,1.921812,10.100026,9.269117
3,1.0,71,2825,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372,5041,7980625,1.206949,10.100026,0.006952,0.000014,3.786566,10.100026,8.354249
4,1.0,71,3376,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438,5041,11397376,1.206949,10.100026,0.006952,0.000013,8.974412,10.100026,2.590290


In [15]:
model.fit(X_squared, y)

model.coef_

array([ 0.00000000e+00,  3.00413966e-09,  1.32794771e-13,  2.43269335e-16,
        1.16448815e-15,  7.02029677e-17, -3.12645606e-17,  1.23380940e-15,
        1.16448815e-15,  1.17773090e-15, -9.11942259e-14,  5.83359899e-13,
        1.29659474e-15,  9.86129203e-15,  6.94445174e-17, -1.85792939e-18,
        8.32757908e-15,  9.86129203e-15,  5.72834172e-15])

In [21]:
corr_1 = X_squared.corr(numeric_only = True)

upper_tri = corr_1.where(np.triu(np.ones(corr_1.shape), k = 1).astype(bool))

high_corr = upper_tri.stack().reset_index()
high_corr.columns = ['Variable 1', 'Variable 2', 'Correlation']

corr_tbl_1 = high_corr[high_corr['Correlation'].ge(0.8)]
corr_tbl_1 = corr_tbl_1.sort_values(by = ['Correlation'], ascending = False)

corr_tbl_1

,Variable 1,Variable 2,Correlation
84,log_num_orders,log_Last Order,1.000000
264,log_num_orders2,log_Last Order2,1.000000
89,log_num_orders,log_num_orders2,0.988527
93,log_num_orders,log_Last Order2,0.988527
165,log_Last Order,log_num_orders2,0.988527
169,log_Last Order,log_Last Order2,0.988527
149,log_Last Product Order,log_Last Product Order2,0.977488
69,log_times_purchased,log_times_purchased2,0.969256
29,user_id,user_id2,0.969009
49,product_id,product_id2,0.968813


Looking at these results, all of the squared values are highly correlated, but all the engineered features are also highly correlated. This is concerning due to multicollinearity. The engineered features might also be highly correlated since they are built in very similar ways

## Dataset 2: Amazon Reviews Dataset

In [23]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked'])
y = data2_transformed['liked']

In [24]:
X.head()

,rating,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,Single-Serve Capsules & Pods,Tea,Cookies,Meat Snacks,"Soups, Stocks & Broths",Nuts & Seeds,Jerky,Juices,Ground Coffee,Breakfast Foods,Cereals,Granola,Fruit & Herbal Tea,Herbal,Bars,Roasted Coffee Beans,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,5,False,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,1,True,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,4,True,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,5,True,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,5,True,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [ ]:
model.fit(X, y)

model.coef_

array([ 0.28319794, -0.00981754,  0.07346078,  0.03150438,  0.02224576,
       -0.19362346, -0.03593603, -0.03095446, -0.02711167, -0.01646754,
        0.01234059,  0.16976443, -0.14671058, -0.03131923, -0.02771575,
        0.01067245, -0.06317729,  0.03860108, -0.00669504,  0.15102118,
       -0.0171856 ,  0.0452093 , -0.04515985,  0.02006723,  0.02006723,
        0.06459393,  0.06371471,  0.06371471, -0.19065059, -0.05069943,
        0.01420402,  0.10095418,  0.01131014,  0.16924854, -0.01189931,
       -0.00131484,  0.00788496])

In [26]:
cols = X.columns

X_squared = pd.concat((X, (X**2).rename(columns = {f'{var}' : f'{var}2' for var in cols })), axis = 1)
X_squared = sm.add_constant(X_squared)

X_squared.head()

,const,rating,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,Single-Serve Capsules & Pods,Tea,Cookies,Meat Snacks,"Soups, Stocks & Broths",Nuts & Seeds,Jerky,Juices,Ground Coffee,Breakfast Foods,Cereals,Granola,Fruit & Herbal Tea,Herbal,Bars,Roasted Coffee Beans,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count,rating2,verified_purchase2,Beverages2,Snacks & Sweets2,Pantry Staples2,Coffee2,Snack Foods2,Candy & Chocolate2,"Bottled Beverages, Water & Drink Mixes2",Cooking & Baking2,Breads & Bakery2,Single-Serve Capsules & Pods2,Tea2,Cookies2,Meat Snacks2,"Soups, Stocks & Broths2",Nuts & Seeds2,Jerky2,Juices2,Ground Coffee2,Breakfast Foods2,Cereals2,Granola2,Fruit & Herbal Tea2,Herbal2,Bars2,Roasted Coffee Beans2,Whole Coffee Beans2,"Canned, Jarred & Packaged Foods2",Cold Cereals2,"Baking Syrups, Sugars & Sweeteners2",Black2,log_helpful_vote2,log_average_rating2,log_price2,log_product_review_count2,log_user_review_count2
0,1.0,5,False,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147,25,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0.000000,2.906166,10.612686,0.480453,0.480453
1,1.0,1,True,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,10.862541,2.843941,11.231256,0.480453,0.480453
2,1.0,4,True,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612,16,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0.000000,2.843941,8.351037,0.480453,1.206949
3,1.0,5,True,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612,25,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,2.967925,7.683782,0.480453,1.206949
4,1.0,5,True,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147,25,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,2.843941,11.527243,2.590290,0.480453


In [27]:
model.fit(X_squared, y)

model.coef_

array([-3.09571524e-15,  2.20819904e-01, -7.55412671e-03,  3.19593698e-02,
        1.20387847e-02,  1.08752389e-02, -8.72293697e-02, -1.76735046e-02,
       -1.38377389e-02, -4.76073900e-03, -9.29937569e-03,  7.47257801e-04,
        7.95763609e-02, -6.92761360e-02, -8.31329786e-03, -1.18984893e-02,
        2.39455060e-03, -2.95063723e-02,  1.73983959e-02,  9.90229667e-03,
        6.95272045e-02, -4.12331783e-03,  6.84306436e-03, -2.66273668e-02,
        1.21089381e-02,  1.21089381e-02,  4.46394986e-02,  3.11486378e-02,
        3.11486378e-02, -9.10065653e-02, -1.55206383e-02,  1.40287860e-02,
        5.19381681e-02,  2.62232630e-02,  1.52900678e+00,  1.69367293e-02,
        2.34562328e-01, -2.77761929e-02,  9.53995736e-03, -7.55412671e-03,
        3.19593698e-02,  1.20387847e-02,  1.08752389e-02, -8.72293697e-02,
       -1.76735046e-02, -1.38377389e-02, -4.76073900e-03, -9.29937569e-03,
        7.47257801e-04,  7.95763609e-02, -6.92761360e-02, -8.31329786e-03,
       -1.18984893e-02,  

In [28]:
corr_2 = X_squared.corr(numeric_only = True)

upper_tri = corr_2.where(np.triu(np.ones(corr_2.shape), k = 1).astype(bool))

high_corr = upper_tri.stack().reset_index()
high_corr.columns = ['Variable 1', 'Variable 2', 'Correlation']

corr_tbl_2 = high_corr[high_corr['Correlation'].ge(0.8)]
corr_tbl_2 = corr_tbl_2.sort_values(by = ['Correlation'], ascending = False)

corr_tbl_2

,Variable 1,Variable 2,Correlation
189,verified_purchase,verified_purchase2,1.000000
4865,Roasted Coffee Beans2,Whole Coffee Beans2,1.000000
265,Beverages,Beverages2,1.000000
341,Snacks & Sweets,Snacks & Sweets2,1.000000
417,Pantry Staples,Pantry Staples2,1.000000
...,...,...,...
839,Breads & Bakery,Cookies,0.850279
1717,Cereals,Cold Cereals2,0.812195
1680,Cereals,Cold Cereals,0.812195
2309,Cold Cereals,Cereals2,0.812195


Again, a lot of the highly correlated features are either features that are duplicates of themselves, the squared version of the feature or the regular features that are correlated. What would be the best move is to remove a lot of the duplicates, and then also figuring out which features should be squares and which should not to prevent multicollinearity.

## Dataset 3: Ecommerce Dataset

In [29]:
data3 = pd.read_csv('/workspaces/DX799---HW-Assignments/data.csv', encoding = 'windows-1252')
data3['Purchased'] = 1

customer_id = data3['Customer ID'].values
segments = data3['Segment'].values
cities = data3['City'].values
state = data3['State'].values
region = data3['Region'].values
product_id = data3['Product ID'].values
categories = data3['Category'].values
subcategories = data3['Sub-Category'].values

def generate_row(df):
    row = {}

    row['Customer ID'] = np.random.choice(customer_id)
    cust_id = row['Customer ID']
    purchased = set(df[df['Customer ID'] == cust_id]['Product ID'])
    candidates = list(set(product_id) - purchased)

    row['Segment'] = np.random.choice(segments)
    row['State'] = np.random.choice(state)
    row['City'] = np.random.choice(cities)
    row['Region'] = np.random.choice(region)
    row['Product ID'] = np.random.choice(candidates)
    row['Category'] = np.random.choice(categories)
    row['Sub-Category'] = np.random.choice(subcategories)

    row['Sales'] = 0
    row['Discount'] = 0
    row['Quantity'] = 0
    row['Profit'] = 0
    row['Purchased'] = 0

    return row

fake_rows = pd.DataFrame([generate_row(data3) for n in range(1000)])

fake_df = pd.concat([data3, fake_rows], ignore_index = True)

fake_df = fake_df.drop(columns = ['Row ID', 'Product Name', 'Country', 'Postal Code', 'Order ID', 'Order Date'])

fake_df_encoded = pd.get_dummies(fake_df, columns = ['Ship Mode', 'Segment', 'State', 'Region', 'Category', 'Sub-Category'], dtype = int)

X = fake_df_encoded.drop(columns = ['Customer ID', 'Product ID', 'City', 'Purchased'])
y = fake_df_encoded['Purchased']

In [30]:
X.head()

,Sales,Quantity,Discount,Profit,Ship Mode_First Class,Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Segment_Consumer,Segment_Corporate,Segment_Home Office,State_Alabama,State_Arizona,State_Arkansas,State_California,State_Colorado,State_Connecticut,State_Delaware,State_District of Columbia,State_Florida,State_Georgia,State_Idaho,State_Illinois,State_Indiana,State_Iowa,State_Kansas,State_Kentucky,State_Louisiana,State_Maryland,State_Massachusetts,State_Michigan,State_Minnesota,State_Mississippi,State_Missouri,State_Montana,State_Nebraska,State_Nevada,State_New Hampshire,State_New Jersey,State_New Mexico,...,State_North Dakota,State_Ohio,State_Oklahoma,State_Oregon,State_Pennsylvania,State_Rhode Island,State_South Carolina,State_South Dakota,State_Tennessee,State_Texas,State_Utah,State_Vermont,State_Virginia,State_Washington,State_West Virginia,State_Wisconsin,Region_Central,Region_East,Region_South,Region_West,Category_Furniture,Category_Office Supplies,Category_Technology,Sub-Category_Accessories,Sub-Category_Appliances,Sub-Category_Art,Sub-Category_Binders,Sub-Category_Bookcases,Sub-Category_Chairs,Sub-Category_Copiers,Sub-Category_Envelopes,Sub-Category_Fasteners,Sub-Category_Furnishings,Sub-Category_Labels,Sub-Category_Machines,Sub-Category_Paper,Sub-Category_Phones,Sub-Category_Storage,Sub-Category_Supplies,Sub-Category_Tables
0,48.896,4,0.2,8.5568,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,474.430,11,0.0,199.2606,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,3.600,2,0.0,1.7280,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,454.560,5,0.2,-107.9580,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
4,141.420,5,0.6,-187.3815,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0


In [31]:
model.fit(X, y)

model.coef_

array([ 6.48668770e-19, -1.19370659e-16,  1.24900090e-15, -3.29597460e-17,
        1.00000000e+00,  1.00000000e+00,  1.00000000e+00,  1.00000000e+00,
       -3.50136586e-14, -3.51663143e-14, -3.51940699e-14,  1.95538030e-14,
        1.89154248e-14,  1.89293026e-14,  2.02060590e-14,  1.88737914e-14,
        1.84574578e-14,  2.01505479e-14,  1.80966353e-14,  1.98729921e-14,
        1.87350135e-14,  1.89848137e-14,  1.94566585e-14,  1.99770755e-14,
        1.96509475e-14,  2.15244489e-14,  1.94289029e-14,  1.84713356e-14,
        1.85684801e-14,  1.94982919e-14,  2.02060590e-14,  1.91929805e-14,
        1.97064587e-14,  1.95468641e-14,  1.85337856e-14,  2.00117700e-14,
        1.80827575e-14,  1.88252192e-14,  1.81799020e-14,  1.84643967e-14,
        1.86239912e-14,  1.93872696e-14,  1.90143040e-14,  1.86101135e-14,
        1.96440086e-14,  1.92276750e-14,  1.82631688e-14,  1.90594068e-14,
        1.96023753e-14,  2.00534034e-14,  2.01123840e-14,  1.90958360e-14,
        1.89744054e-14,  

In [32]:
cols = X.columns

X_squared = pd.concat((X, (X**2).rename(columns = {f'{var}' : f'{var}2' for var in cols })), axis = 1)
X_squared = sm.add_constant(X_squared)

X_squared.head()

,const,Sales,Quantity,Discount,Profit,Ship Mode_First Class,Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Segment_Consumer,Segment_Corporate,Segment_Home Office,State_Alabama,State_Arizona,State_Arkansas,State_California,State_Colorado,State_Connecticut,State_Delaware,State_District of Columbia,State_Florida,State_Georgia,State_Idaho,State_Illinois,State_Indiana,State_Iowa,State_Kansas,State_Kentucky,State_Louisiana,State_Maryland,State_Massachusetts,State_Michigan,State_Minnesota,State_Mississippi,State_Missouri,State_Montana,State_Nebraska,State_Nevada,State_New Hampshire,State_New Jersey,...,State_North Dakota2,State_Ohio2,State_Oklahoma2,State_Oregon2,State_Pennsylvania2,State_Rhode Island2,State_South Carolina2,State_South Dakota2,State_Tennessee2,State_Texas2,State_Utah2,State_Vermont2,State_Virginia2,State_Washington2,State_West Virginia2,State_Wisconsin2,Region_Central2,Region_East2,Region_South2,Region_West2,Category_Furniture2,Category_Office Supplies2,Category_Technology2,Sub-Category_Accessories2,Sub-Category_Appliances2,Sub-Category_Art2,Sub-Category_Binders2,Sub-Category_Bookcases2,Sub-Category_Chairs2,Sub-Category_Copiers2,Sub-Category_Envelopes2,Sub-Category_Fasteners2,Sub-Category_Furnishings2,Sub-Category_Labels2,Sub-Category_Machines2,Sub-Category_Paper2,Sub-Category_Phones2,Sub-Category_Storage2,Sub-Category_Supplies2,Sub-Category_Tables2
0,1.0,48.896,4,0.2,8.5568,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,1.0,474.430,11,0.0,199.2606,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,1.0,3.600,2,0.0,1.7280,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1.0,454.560,5,0.2,-107.9580,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
4,1.0,141.420,5,0.6,-187.3815,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0


In [33]:
model.fit(X_squared, y)

model.coef_

array([ 3.28825368e-16,  2.13288194e-04,  5.80458807e-04,  7.56173135e-06,
        8.07818151e-05,  5.10326406e-06,  7.58159564e-07,  8.62428156e-06,
        1.98373056e-05,  3.09061501e-06, -1.07997517e-06, -2.01063984e-06,
       -1.94674017e-08,  5.24035496e-07,  1.58125040e-07,  4.38673551e-06,
        3.58614596e-07, -6.23800932e-07,  2.05691283e-07,  4.55424333e-08,
       -1.73557502e-06, -4.92834618e-07, -1.47110873e-07,  4.51665471e-07,
       -2.22191346e-07, -1.48422080e-07, -5.32308965e-08, -4.81253380e-07,
       -1.68841131e-07,  4.45171159e-08, -4.19451676e-07, -3.32184745e-07,
       -3.22993545e-07,  1.62348357e-07,  4.97450855e-08, -1.90047077e-07,
       -3.44697846e-07,  5.34287638e-08,  3.38572553e-07, -5.97099541e-07,
        3.70849899e-07, -5.64106053e-07,  2.80337190e-07,  1.90503048e-07,
        5.93593421e-07, -1.90233794e-07,  8.99389906e-08, -2.69545087e-07,
        3.39584853e-08,  1.70756108e-07, -1.69368635e-08,  2.73490943e-07,
       -3.21960211e-07,  

In [34]:
corr_3 = X_squared.corr(numeric_only = True)

upper_tri = corr_3.where(np.triu(np.ones(corr_3.shape), k = 1).astype(bool))

high_corr = upper_tri.stack().reset_index()
high_corr.columns = ['Variable 1', 'Variable 2', 'Correlation']

corr_tbl_3 = high_corr[high_corr['Correlation'].ge(0.8)]
corr_tbl_3 = corr_tbl_3.sort_values(by = ['Correlation'], ascending = False)

corr_tbl_3

,Variable 1,Variable 2,Correlation
912,Ship Mode_First Class,Ship Mode_First Class2,1.000000
2738,State_Colorado,State_Colorado2,1.000000
1078,Ship Mode_Same Day,Ship Mode_Same Day2,1.000000
1244,Ship Mode_Second Class,Ship Mode_Second Class2,1.000000
1410,Ship Mode_Standard Class,Ship Mode_Standard Class2,1.000000
...,...,...,...
13196,Sub-Category_Phones,Sub-Category_Phones2,1.000000
13694,Sub-Category_Tables,Sub-Category_Tables2,1.000000
13781,Sales2,Profit2,0.931101
580,Discount,Discount2,0.928333


Again, the highly correlated features are the ones that are squares of the regular features and the features that are similar or related.